# Semana 13 - Validação e Otimização de Modelos (Titanic)


In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [7]:
# Carregar Dataset
path_dataset = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'

df = pd.read_csv(path_dataset)

# Visualizar as primeiras linhas
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [18]:
# Explorar as características do Dataset
df.info()

df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,177
SibSp,0
Parch,0
Ticket,0
Fare,0


In [19]:
# Tratamento de Nulos
df['Age'] = df['Age'].fillna(df['Age'].median())
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

# Padronizando variáveis categóricas
df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})
df['Embarked'] = df['Embarked'].map({'S': 0, 'C': 1, 'Q': 2})

# Remover colunas indesejadas
df.drop(['PassengerId', 'Name', 'Ticket', 'Cabin'], axis=1, inplace=True)

# Visualizar as primeiras linhas
df.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,0,22.0,1,0,7.2500,0
1,1,1,1,38.0,1,0,71.2833,1
2,1,3,1,26.0,0,0,7.9250,0
3,1,1,1,35.0,1,0,53.1000,0
4,0,3,0,35.0,0,0,8.0500,0


In [21]:
# Separar treino e teste

X = df.drop('Survived', axis=1)
y = df['Survived']

x_treino, x_teste, y_treino, y_teste = train_test_split(X, y, test_size=0.2, random_state=42)

In [22]:
# Testando no modelo padrão

modelo = RandomForestClassifier(random_state=42)
modelo.fit(x_treino, y_treino)
y_pred = modelo.predict(x_teste)

standar_accuracy = accuracy_score(y_teste, y_pred)
print(f'Acurácia: {standar_accuracy:.4f}')
print(classification_report(y_teste, y_pred))

Acurácia: 0.8268
              precision    recall  f1-score   support

           0       0.84      0.88      0.86       105
           1       0.81      0.76      0.78        74

    accuracy                           0.83       179
   macro avg       0.82      0.82      0.82       179
weighted avg       0.83      0.83      0.83       179



In [23]:
# Utilizando o GridSearch
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 5, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

grid = GridSearchCV(modelo, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid.fit(x_treino, y_treino)

GridSearchCV(cv=5, estimator=RandomForestClassifier(random_state=42), n_jobs=-1,
             param_grid={'max_depth': [None, 5, 10, 20],
                         'min_samples_leaf': [1, 2, 4],
                         'min_samples_split': [2, 5, 10],
                         'n_estimators': [100, 200, 300]},
             scoring='accuracy')

In [26]:
# Resultados do GridSearch
print(f'Melhor Acurácia: {grid.best_score_:.4f}')
print(f'Melhor configuração: {grid.best_params_}')


Melhor Acurácia: 0.8357
Melhor configuração: {'max_depth': 5, 'min_samples_leaf': 2, 'min_samples_split': 10, 'n_estimators': 100}


In [27]:
# Aplicação dos resultados à um novo modelo
modelo_otimizado = grid.best_estimator_
pred_grid = modelo_otimizado.predict(x_teste)

grid_accuracy = accuracy_score(y_teste, pred_grid)
print(f'Acurácia: {grid_accuracy:.4f}')
print(classification_report(y_teste, pred_grid))

Acurácia: 0.8045
              precision    recall  f1-score   support

           0       0.80      0.89      0.84       105
           1       0.81      0.69      0.74        74

    accuracy                           0.80       179
   macro avg       0.81      0.79      0.79       179
weighted avg       0.80      0.80      0.80       179



In [29]:
# Comparando os resultados dos modelos
print(f'Acurácia padrão: {standar_accuracy:.4f}')
print(f'Acurácia otimizada: {grid_accuracy:.4f}')
print(f'Diferença: {grid_accuracy - standar_accuracy:.4f}')

Acurácia padrão: 0.8268
Acurácia otimizada: 0.8045
Diferença: -0.0223


In [31]:
# Testando outras amostras

for cv in [3,5,10, 20]:
  grid = GridSearchCV(modelo, param_grid, cv=cv, scoring='accuracy', n_jobs=-1)
  grid.fit(x_treino, y_treino)
  print(f"CV: {cv} - Melhor Acurácia: {grid.best_score_:.4f}")

CV: 3 - Melhor Acurácia: 0.8329
CV: 5 - Melhor Acurácia: 0.8357
CV: 10 - Melhor Acurácia: 0.8357
CV: 20 - Melhor Acurácia: 0.8370


Perguntas:

1. Quais foram os melhores hiperparâmetros encontrados?
Os melhores parâmetros encontrados foram:
'max_depth': 5, 'min_samples_leaf': 2, 'min_samples_split': 10, 'n_estimators': 100.

2. O modelo otimizado teve uma acurácia melhor que o modelo padrão?
Quanto?
O modelo otimizado teve uma acurácia pior do que o modelo padrão, obtivemos:
Acurácia padrão: 0.8268 e Acurácia otimizada: 0.8045

3. O que aconteceu quando você mudou o número de folds no Cross-
Validation?
Obtivemos uma melhora na acurácia, com 3 folds a acurácia foi de 0.8329, com 5 folds a acurácia foi de 0.8357, com 10 folds a acurácia permaneceu em 0.8357 e com 20 folds a acurácia aumentou para 0.8370. Ou seja, atingimos uma boa acurácia com 5 folds sem demandar muito tempo.

4. Você acha que o GridSearch compensa o tempo de processamento? Por quê?
Apesar de não melhorarmos a acurácia com os parâmetros encontrados pelo GridSearch, é inegável que o uso fornece praticidade de agilidade no teste, então eu considero que compensa o tempo.